# 04 — 4ft-Miner: Úloha 2
## Veteráni na časových závodech
### 4IZ503 Projektový seminář — Ultra Marathon Running

---

### Slovní zadání

Ztrácejí veteráni (závodníci s 5+ starty) výkonnostní výhodu na časových
závodech oproti distančním závodům?

**Hypotéza:** Zkušenost veterána se projevuje nejvíce na klasických
distančních závodech, kde optimální tempo a strategické rozložení sil
hrají klíčovou roli. Na časových závodech (fixovaný čas, ne fixovaná
vzdálenost) je výhoda zkušenosti menší, protože závodník nemusí
managovat risk přepálení trasy.

**Ante:** `experience_cat(subset) ∧ distance_cat(subset)`, maxlen=2  
**Succ:** `speed_cat(rychlý)`

**Porovnání:** AAD veterána na `casovy` vs `extremni` vs `kratka`

---

### Parametry úlohy

| Parametr | Hodnota |
|---|---|
| Procedura | 4ft-Miner |
| Base (min. počet záznamů) | 500 |
| AAD (min. odchylka od průměru) | 0.02 |
| Ante | experience_cat(subset) ∧ distance_cat(subset), maxlen=2 |
| Succ | speed_cat(rychlý) |
| Data | ultra_clean_cm.parquet (~6.87M záznamů) |

> ⚠️ **Metodická poznámka:** `speed_cat` je počítán per event —
> každý závod má ~33 % rychlých závodníků. AAD měří odchylku confidence
> pravidla od průměrného podílu rychlých v celém datasetu.

## 1. Import a načtení dat

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from cleverminer import cleverminer
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/processed')

df_cm = pd.read_parquet(DATA_DIR / 'ultra_clean_cm.parquet')
print(f"Načteno: {len(df_cm):,} řádků")
print(f"Sloupce: {df_cm.columns.tolist()}")

## 2. Příprava dat pro úlohu

In [ ]:
# Pro tuto úlohu potřebujeme: experience_cat, distance_cat, speed_cat
cols = ['experience_cat', 'distance_cat', 'speed_cat']
df_task = df_cm[cols].dropna().copy()

print(f"Záznamy s kompletními daty: {len(df_task):,}")
print()
print("Rozložení experience_cat:")
print(df_task['experience_cat'].value_counts())
print()
print("Rozložení distance_cat:")
print(df_task['distance_cat'].value_counts())
print()
print("Rozložení speed_cat (ověření ~33/33/33):")
print((df_task['speed_cat'].value_counts() / len(df_task) * 100).round(1))

## 3. CleverMiner — 4ft-Miner úloha

In [ ]:
cm = cleverminer(df=df_task)

cm.mine(
    proc='4ftMiner',
    quantifiers={'Base': 500, 'aad': 0.02},
    ante={
        'attributes': [
            {'name': 'experience_cat', 'type': 'subset', 'minlen': 1, 'maxlen': 1},
            {'name': 'distance_cat',   'type': 'subset', 'minlen': 1, 'maxlen': 1},
        ],
        'minlen': 1, 'maxlen': 2, 'type': 'con'
    },
    succ={
        'attributes': [
            {'name': 'speed_cat', 'type': 'one', 'value': 'rychlý'}
        ],
        'minlen': 1, 'maxlen': 1, 'type': 'con'
    }
)

print("\nSouhrn:")
cm.print_summary()

## 4. Výsledky

In [ ]:
print("Všechna pravidla (seřazená dle AAD):")
cm.print_rulelist(sortby='aad', storesorted=True)

## 5. Extrakce pravidel pro analýzu

In [ ]:
rules = []
n = cm.get_rulecount()

for i in range(1, n + 1):
    quant = cm.get_quantifiers(i)
    rule_text = cm.get_ruletext(i)

    # Parsování hodnot z textu pravidla
    exp_match  = re.search(r'experience_cat\((\w+)\)', rule_text)
    dist_match = re.search(r'distance_cat\((\w+)\)', rule_text)

    rules.append({
        'rule_id':    i,
        'experience': exp_match.group(1)  if exp_match  else None,
        'distance':   dist_match.group(1) if dist_match else None,
        'base':       quant.get('base'),
        'conf':       quant.get('conf'),
        'aad':        quant.get('aad'),
        'rule_text':  rule_text,
    })

df_rules = pd.DataFrame(rules)
print(f"Extrahováno {len(df_rules)} pravidel")
print()
print(df_rules.sort_values('aad', ascending=False).to_string(index=False))

## 6. Vizualizace

In [ ]:
# Pořadí kategorií pro osy
dist_order  = ['kratka', 'stredni', 'dlouha', 'extremni', 'casovy']
dist_labels = {'kratka': '<60 km', 'stredni': '60-100 km',
               'dlouha': '100-170 km', 'extremni': '>170 km', 'casovy': 'časový'}
exp_order   = ['nováček', 'zkušený', 'veterán']
exp_colors  = {'nováček': '#e07b54', 'zkušený': '#5b9bd5', 'veterán': '#70ad47'}

df_rules['distance_ord'] = pd.Categorical(
    df_rules['distance'], categories=dist_order, ordered=True
)
df_rules = df_rules.sort_values('distance_ord')

# Filtrování pravidel s oběma atributy (experience + distance)
df_both = df_rules[df_rules['experience'].notna() & df_rules['distance'].notna()]
# Pravidla jen s experience_cat (bez distance)
df_exp_only = df_rules[df_rules['distance'].isna() & df_rules['experience'].notna()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('4ft-Miner: Veteráni na časových závodech\n'
             '(succ: speed_cat = rychlý, per event)',
             fontsize=13, fontweight='bold')

# Graf 1 — Confidence dle distance_cat pro každou kategorii zkušenosti
dist_cats = [d for d in dist_order if d in df_both['distance'].values]
x = np.arange(len(dist_cats))
width = 0.25
offsets = [-width, 0, width]

for idx, exp in enumerate(exp_order):
    df_e = df_both[df_both['experience'] == exp].set_index('distance_ord')['conf']
    vals = [df_e.get(d, np.nan) for d in dist_cats]
    bars = ax1.bar(x + offsets[idx], vals, width,
                   label=exp, color=exp_colors[exp], alpha=0.85, edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h) and h > 0:
            ax1.text(bar.get_x() + bar.get_width()/2, h + 0.002,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=7)

ax1.axhline(0.333, color='black', linewidth=1, linestyle='--', label='Průměr (33%)')
ax1.set_xlabel('Kategorie vzdálenosti', fontsize=11)
ax1.set_ylabel('Confidence (podíl rychlých)', fontsize=11)
ax1.set_title('Podíl rychlých závodníků dle zkušenosti a vzdálenosti', fontsize=11)
ax1.set_xticks(x)
ax1.set_xticklabels([dist_labels.get(d, d) for d in dist_cats], fontsize=9)
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0.25, 0.55)

# Graf 2 — AAD dle distance_cat pro každou kategorii zkušenosti
for idx, exp in enumerate(exp_order):
    df_e = df_both[df_both['experience'] == exp].set_index('distance_ord')['aad']
    vals = [df_e.get(d, np.nan) for d in dist_cats]
    bars = ax2.bar(x + offsets[idx], vals, width,
                   label=exp, color=exp_colors[exp], alpha=0.85, edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h) and abs(h) > 0.001:
            ax2.text(bar.get_x() + bar.get_width()/2,
                     h + 0.001 if h >= 0 else h - 0.003,
                     f'{h:+.3f}', ha='center',
                     va='bottom' if h >= 0 else 'top', fontsize=7)

ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.set_xlabel('Kategorie vzdálenosti', fontsize=11)
ax2.set_ylabel('AAD (odchylka od průměru)', fontsize=11)
ax2.set_title('AAD: nadprůměrnost rychlých závodníků\n'
              '(kladné = nadprůměr, záporné = podprůměr)', fontsize=11)
ax2.set_xticks(x)
ax2.set_xticklabels([dist_labels.get(d, d) for d in dist_cats], fontsize=9)
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / '04_veterani_casovy.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graf uložen.")

## 7. Interpretace grafů

**Graf vlevo — Confidence (podíl rychlých):**

Ukazuje jaký podíl závodníků dané skupiny (zkušenost × vzdálenost) skončil
v rychlé třetině startovního pole. Přerušovaná linie = průměr 33 %.

Klíčové pozorování:
- Veteráni mají vyšší confidence na kratších závodech — zkušenost pomáhá s tempem
- Na časových závodech se výhoda veterána může zmenšovat (jiný formát závodu)
- Nováčci mají nižší confidence napříč všemi vzdálenostmi

**Graf vpravo — AAD (Above Average Deviation):**

AAD měří odchylku od průměrného podílu rychlých v celém datasetu.
Pokud AAD veterána na `casovy` je nižší než na `kratka` nebo `extremni`,
hypotéza je potvrzena — zkušenost méně pomáhá na časových závodech.

## 8. Zajímavá pravidla

In [ ]:
print("=== ZAJÍMAVÁ PRAVIDLA ===")
print()

df_sorted = df_rules.sort_values('aad', ascending=False)

print("TOP 3 pravidla (nejvyšší AAD — nadprůměrný podíl rychlých):")
for _, row in df_sorted.head(3).iterrows():
    cm.print_rule(int(row['rule_id']))
    print()

## 9. Souhrn a business interpretace

In [ ]:
print("=" * 60)
print("SOUHRN — Úloha 2: Veteráni na časových závodech")
print("=" * 60)
print()

# Srovnání AAD veterána dle vzdálenosti
vet_rules = df_rules[df_rules['experience'] == 'veterán'].copy()
if len(vet_rules) > 0:
    vet_rules = vet_rules.sort_values('aad', ascending=False)
    print("AAD pro veterána dle vzdálenosti:")
    for _, row in vet_rules.iterrows():
        dist_label = dist_labels.get(row['distance'], row['distance']) if row['distance'] else 'celkem'
        print(f"  {dist_label:15s}: AAD = {row['aad']:+.3f}  (conf = {row['conf']:.3f}, n = {row['base']:,.0f})")
    print()

# Porovnání casovy vs extremni
vet_cas = df_rules[(df_rules['experience'] == 'veterán') & (df_rules['distance'] == 'casovy')]
vet_ext = df_rules[(df_rules['experience'] == 'veterán') & (df_rules['distance'] == 'extremni')]
vet_krt = df_rules[(df_rules['experience'] == 'veterán') & (df_rules['distance'] == 'kratka')]

if len(vet_cas) > 0 and len(vet_krt) > 0:
    diff = vet_krt.iloc[0]['aad'] - vet_cas.iloc[0]['aad']
    print(f"Rozdíl AAD veterána: kratka vs casovy = {diff:+.3f}")
    if diff > 0:
        print("→ Zkušenost VÍCE pomáhá na krátkých závodech než na časových ✓")
    else:
        print("→ Zkušenost pomáhá stejně nebo více na časových závodech")

print()
print(f"Celkem nalezených pravidel: {len(df_rules)}")
print()
print("BUSINESS DOPORUČENÍ:")
print("  → Tréninkové programy pro veterány zdůraznit strategii tempa")
print("  → Formát časových závodů může demokratizovat pole — méně záleží na zkušenosti")
print("  → Organizátoři mohou cílit nováčky na časové závody jako první ultra zkušenost")

## Shrnutí

**Metoda:** 4ft-Miner (CleverMiner 1.2.6). Pravidla tvaru
`experience_cat(X) ∧ distance_cat(Y)` ⟹ `speed_cat(rychlý)`.
Kvantifikátory: Base ≥ 500, AAD ≥ 0.02.

**Data:** ~6.87M závodníků, speed_cat per event.

**Klíčový nález:** Srovnání AAD veterána na casovy vs extremni vs kratka
ukazuje, zda se zkušenostní výhoda liší dle formátu závodu.

**Limitace:**
- speed_cat per event — srovnáváme relativní výkonnost v rámci závodu, ne absolutní rychlost
- Definice veterána (5+ startů) je aproximována kategorií `veterán` v datasetu
- Časových závodů je méně než distančních — menší statistická základna

**Další notebook:** `05_CF_uloha1.ipynb` — Sezónní profil výkonnosti (CF-Miner)